# Quantum Fourier Transform

The QFT transforms between computational and frequency bases.
It is the quantum analogue of the discrete Fourier Transform.

In [ ]:
import cirq
import numpy as np

## QFT Circuit (3 qubits)

H gates + controlled phase rotations + SWAP reversal.

In [ ]:
def qft_circuit(qubits):
    n = len(qubits)
    ops = []
    for i in range(n):
        ops.append(cirq.H(qubits[i]))
        for j in range(i + 1, n):
            k = j - i
            ops.append(cirq.CZPowGate(exponent=1.0 / (2**k))(qubits[j], qubits[i]))
    ops.extend(cirq.SWAP(qubits[i], qubits[n - 1 - i]) for i in range(n // 2))
    return cirq.Circuit(ops)

qubits = cirq.LineQubit.range(3)
print(qft_circuit(qubits))

## Apply QFT to |101\u27e9

In [ ]:
sim = cirq.Simulator()
q0, q1, q2 = cirq.LineQubit.range(3)

prep = cirq.Circuit(cirq.X(q0), cirq.X(q2))
qft = qft_circuit(cirq.LineQubit.range(3))

result = sim.simulate(prep + qft)
sv = result.final_state_vector
print("QFT|101\u27e9:")
for i in range(8):
    if abs(sv[i]) > 1e-6:
        print(f"  |{i:03b}\u27e9: {sv[i]:.4f}")

## Roundtrip: QFT then Inverse QFT

In [ ]:
def iqft_circuit(qubits):
    n = len(qubits)
    ops = []
    ops.extend(cirq.SWAP(qubits[i], qubits[n - 1 - i]) for i in range(n // 2))
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            k = j - i
            ops.append(cirq.CZPowGate(exponent=-1.0 / (2**k))(qubits[j], qubits[i]))
        ops.append(cirq.H(qubits[i]))
    return cirq.Circuit(ops)

qubits = cirq.LineQubit.range(3)
original = cirq.Circuit(cirq.X(qubits[0]), cirq.X(qubits[2]))
roundtrip = original + qft_circuit(qubits) + iqft_circuit(qubits)
sv = sim.simulate(roundtrip).final_state_vector
print("Original |101\u27e9:", np.array2string(sim.simulate(original).final_state_vector, precision=3))
print("After QFT->IQFT:", np.array2string(sv, precision=3))